# 2.回答回退策略检索器

知识点讲解：Step-Back Prompting 回答回退策略

1. 过于具体的问题为什么检索不到
   - 用户提问往往包含大量具体限定词，例如"人工智能会让世界发生翻天覆地的变化吗？"
   - 这类问题的向量带有强烈的"具体表述特征"，而知识库中的文档通常是
     概念性、原理性的通用描述（如"人工智能的发展趋势与社会影响"）
   - 两者向量距离较远，导致明明有相关资料却召回失败
   - 另一种典型场景：问题中含有知识库不存在的专有名词，向量被这个词"带偏"

2. Step-Back Prompting 的核心思想
   - 源自 Google DeepMind 论文《Take a Step Back: Evoking Reasoning through
     Abstraction in Large Language Models》
   - 思路：先"后退一步"，把具体问题抽象为更宽泛的前置问题（Step-Back Question），
     用这个更通用的问题去检索，从而命中知识库中的概念性文档
   - 类比人类思维：回答"笨笨这只猫为什么爱睡觉"之前，
     先想"猫的睡眠习性是怎样的"，从一般原理推导具体答案

3. 抽象方向的三种典型模式（本例示例体现得很清楚）
   - 范围放大："慕课网上有关于AI应用开发的课程吗？" → "慕课网上有哪些课程？"
     （从具体品类退到全集）
   - 维度上升："慕小课出生在哪个国家？" → "慕小课的人生经历是什么样的？"
     （从单一属性退到整体背景）
   - 约束去除："司机可以开快车吗？" → "司机可以做什么？"
     （从特定行为退到行为集合）

4. 为什么必须用 Few-Shot 来实现问题回退
   - "把问题抽象化"是一个高度依赖语感的任务，纯文字指令很难描述清楚
     "抽象到什么程度才合适"——退得太少没效果，退得太多就跑题了
   - 通过 3 个示例，模型能直接类比出合适的抽象粒度
   - 这正是上一个示例（1.少量示例提示模板.py）的实际应用

5. 本例检索器的链式设计（注意最后一环）
     {"question": RunnablePassthrough()} | prompt | llm | StrOutputParser() | self.retriever
   - 前四环生成回退问题的字符串
   - 关键点：最后一环直接把 retriever 接入管道！
     因为 BaseRetriever 实现了 Runnable，输入 str 输出 list[Document]，
     正好能承接 StrOutputParser 的字符串输出
   - 所以 chain.invoke(query) 的返回值直接就是 List[Document]，
     一条管道同时完成"改写 + 检索"，代码极其简洁

6. Step-Back 与其他策略的区别
   - Multi-Query：横向扩展，生成多个同层级的等价查询（广度）
   - Step-Back：纵向抽象，生成一个更高层级的问题（深度/高度）
   - 问题分解：向下拆解，生成多个更细的子问题（与 Step-Back 方向相反）
   - HyDE：生成假设性答案文档，做 doc-doc 对称检索（形态转换）
   - 四者可以组合使用，例如先 Step-Back 再 Multi-Query

7. 本例实现的一个局限
   - 只用"回退问题"检索，完全丢弃了原始问题的检索结果
   - 论文中的标准做法是"原始问题 + 回退问题"都检索，然后合并，
     既保留具体细节的命中，又补充概念性背景
   - 生产环境建议改为双路召回 + RRF 融合

8. 最佳实践建议
   - 回退问题的示例要覆盖多种抽象方向（放大范围、上升维度、去除约束），
     示例质量直接决定改写效果
   - 生成回退问题的 LLM 必须 temperature=0，保证同一问题的回退结果稳定
   - 推荐改为"原始问题 + 回退问题"双路检索并用 RRF 融合，避免丢失具体细节
   - 不要过度回退：退得太宽泛会召回大量无关文档，反而降低精度；
     可在 system 提示中加入"保持与原问题同一领域"的约束
   - 适用场景：概念性提问、含知识库不存在的专有名词、用户表述过于具体；
     不适用：精确参数查询、ID/编号查找（回退会丢失关键限定词）
   - 本示例在 _get_relevant_documents 内部每次都重建 Prompt 与链，
     属于教学写法；生产环境应在 __init__ 或类级别构建一次以复用，减少开销
   - 内部调用 self.retriever 时建议传 config={"callbacks": run_manager.get_child()}，
     保持 LangSmith 上的调用链完整可追踪


In [ ]:
from typing import List
import dotenv
import weaviate
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.documents import Document
from langchain_core.language_models import BaseLanguageModel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.retrievers import BaseRetriever
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_weaviate import WeaviateVectorStore
from weaviate.auth import AuthApiKey
dotenv.load_dotenv()

In [ ]:
class StepBackRetriever(BaseRetriever):
    """回答回退检索器
    继承 BaseRetriever 实现自定义检索逻辑：
    先用 LLM 把具体问题抽象为更宽泛的前置问题，再用该问题执行向量检索。
    """
    retriever: BaseRetriever
    llm: BaseLanguageModel
    def _get_relevant_documents(
            self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> List[Document]:
        """根据传递的query执行问题回退并检索
        参数：
            query: str - 用户的原始（通常较具体的）问题
            run_manager: CallbackManagerForRetrieverRun - 回调管理器（关键字参数）
        返回：
            List[Document] - 用回退问题检索到的文档列表
        执行流程：
            1) 构建 Few-Shot 示例，教会 LLM 如何做问题抽象
            2) 组装完整 Prompt（system 指令 + 示例 + 真实问题）
            3) 构建 "生成回退问题 → 检索" 的一体化链
            4) 执行链并直接返回文档列表
        """
        examples = [
            {"input": "慕课网上有关于AI应用开发的课程吗？", "output": "慕课网上有哪些课程？"},
            {"input": "慕小课出生在哪个国家？", "output": "慕小课的人生经历是什么样的？"},
            {"input": "司机可以开快车吗？", "output": "司机可以做什么？"},
        ]
        example_prompt = ChatPromptTemplate.from_messages([
            ("human", "{input}"),
            ("ai", "{output}"),
        ])
        few_shot_prompt = FewShotChatMessagePromptTemplate(
            examples=examples,
            example_prompt=example_prompt,
        )
        prompt = ChatPromptTemplate.from_messages([
            ("system",
             "你是一个世界知识的专家。你的任务是回退问题，将问题改述为更一般或者前置问题，这样更容易回答，请参考示例来实现。"),
            few_shot_prompt,
            ("human", "{question}"),
        ])
        chain = (
                {"question": RunnablePassthrough()}
                | prompt
                | self.llm
                | StrOutputParser()
                | self.retriever
        )
        return chain.invoke(query)

In [ ]:
db = WeaviateVectorStore(
    client=weaviate.connect_to_wcs(
        cluster_url="https://mbakeruerziae6psyex7ng.c0.us-west3.gcp.weaviate.cloud",
        auth_credentials=AuthApiKey("ZltPVa9ZSOxUcfafelsggGyyH6tnTYQYJvBx"),
    ),
    index_name="DatasetDemo",
    text_key="text",
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
)
retriever = db.as_retriever(search_type="mmr")
step_back_retriever = StepBackRetriever(
    retriever=retriever,
    llm=ChatOpenAI(model="deepseek-v4-pro", temperature=0),
)
documents = step_back_retriever.invoke("人工智能会让世界发生翻天覆地的变化吗？")
print(documents)
print(len(documents))